Bronze data processing

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/akhildataengineer@hotmail.com/Atlikon_migration_pipeline/1_setup/3_utilities

In [0]:
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "gross_price", "Data Source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f's3://sportsbar-store-dp1/{data_source}/*.csv' #s3://sportsbar-store-dp/customers/*.csv
base_path

In [0]:
dbutils.fs.ls('s3://sportsbar-store-dp1/')

In [0]:
df_raw = spark.read\
        .format('csv')\
        .option('header', True)\
        .option('inferSchema', True)\
        .load(base_path)\
        .withColumn("read_timestamp", F.current_timestamp())\
        .select("*", "_metadata.file_name", "_metadata.file_size")

display(df_raw.limit(10))

In [0]:
df_raw.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

Silver Data Processing

In [0]:
df_enr = spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.{data_source}")
display(df_enr.limit(10))

In [0]:
# Normalize the month table
date_formats = ["yyyy/MM/dd", "dd/MM/yyyy", "yyyy-MM-dd", "dd-MM-yyyy"]

df_enr = df_enr.withColumn("month", F.coalesce(
    F.try_to_date(F.col("month"), "yyyy/MM/dd"),
    F.try_to_date(F.col("month"), "dd/MM/yyyy"),
    F.try_to_date(F.col("month"), "yyyy-MM-dd"),
    F.try_to_date(F.col("month"), "dd-MM-yyyy")
))

In [0]:
# Cleaning gross price
df_enr = df_enr.withColumn("gross_price", F.when(F.col("gross_price").rlike(r"[0-9]"), F.regexp_replace(F.col("gross_price"), "[^0-9.]", "").cast("double"))\
                                           .otherwise(0))

In [0]:
# Join products table with this 
df_prod = spark.table(f"{catalog}.{silver_schema}.products")
df_joined = df_enr.join(df_prod.select("product_id", "product_code"), on="product_id", how="inner")
df_joined = df_joined.select("product_id", "product_code", "month", "gross_price", "read_timestamp", "file_name", "file_size")

df_joined.show()

In [0]:
df_joined.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .option("mergeSchema", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

Gold Data Processing

In [0]:
df_agg = spark.sql(f"SELECT * FROM {catalog}.{silver_schema}.{data_source}")
df_agg = df_agg.select("product_code", "month", "gross_price")

In [0]:
df_agg.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", 'true')\
    .option("mergeSchema", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")

Merge to the parent table

In [0]:
# 
df_agg = spark.sql(f"SELECT product_code, month, gross_price FROM {catalog}.{gold_schema}.sb_dim_{data_source}")
df_agg.show()

In [0]:
df_agg = df_agg.withColumn("year", F.year("month"))\
               .withColumn("isZero", F.when(F.col("gross_price") == 0, 1).otherwise(0))

win_fn = Window.partitionBy("product_code", "year").orderBy(F.col("isZero"), F.col("month").desc())
df_agg = df_agg.withColumn("rank", F.row_number().over(win_fn)).filter(F.col("rank") == 1)

In [0]:
# Upsert
delta_table = DeltaTable.forName(spark, f"{catalog}.{gold_schema}.dim_{data_source}")

delta_table.alias("t")\
        .merge(
            source = df_agg.alias("s"),
            condition = "t.product_code = s.product_code"
        )\
        .whenMatchedUpdate(
            set = {
                "t.price_inr": "s.gross_price",
                "t.year": "s.year"
            }
        )\
        .whenNotMatchedInsert(
            values = {
                "t.product_code": "s.product_code",
                "t.price_inr": "s.gross_price",
                "t.year": "s.year"
            }
        )\
        .execute()